# 05_05 - Weighted Blending

Run weighted blends across available Phase 5 base model probabilities.


## Contract

Reads probabilities from `05_01` to `05_04`. Writes sweep, best blend metrics and selected blend probabilities.

- **Multi-seed:** loops over `SEEDS`; reads base probabilities from the matching seed subfolder before blending.


In [1]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except ImportError:
except ImportError:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 tr...: ném lỗi và dừng cell
    raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 training locally.")


RuntimeError: Run this notebook in Google Colab. Do not execute Phase 5 training locally.

In [ ]:
# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount("/content/drive"): mount Google Drive trên Colab
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# import gc: giải phóng bộ nhớ
import gc
# import importlib: import thư viện importlib
import importlib
# import json: đọc/ghi JSON metadata
import json
# import os: biến môi trường hệ thống
import os
# import random: cố định seed ngẫu nhiên
import random
# import subprocess: chạy lệnh pip/cài package
import subprocess
# import sys: tham số Python runtime
import sys
# import time: đo thời gian thực thi
import time
# from datetime import datetime, timezone: import thư viện datetime
from datetime import datetime, timezone
# from itertools import combinations, product: import thư viện itertools
from itertools import combinations, product
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import joblib: lưu/tải object đã fit
import joblib
# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# from sklearn.metrics import (: thư viện machine learning scikit-learn
from sklearn.metrics import (
    # accuracy_score,: thực thi lệnh Python
    accuracy_score,
    # average_precision_score,: thực thi lệnh Python
    average_precision_score,
    # brier_score_loss,: thực thi lệnh Python
    brier_score_loss,
    # confusion_matrix,: thực thi lệnh Python
    confusion_matrix,
    # f1_score,: thực thi lệnh Python
    f1_score,
    # precision_recall_fscore_support,: thực thi lệnh Python
    precision_recall_fscore_support,
    # roc_auc_score,: thực thi lệnh Python
    roc_auc_score,
# ): đóng ngoặc gọi hàm
)

# SEEDS: biến cấu hình/hằng số của notebook
SEEDS = [42, 123, 456]
# SEED: biến cấu hình/hằng số của notebook
SEED = SEEDS[0]
# FAKE_LABEL: biến cấu hình/hằng số của notebook
FAKE_LABEL = 1
# REAL_LABEL: biến cấu hình/hằng số của notebook
REAL_LABEL = 0
# DEFAULT_THRESHOLD: biến cấu hình/hằng số của notebook
DEFAULT_THRESHOLD = 0.50
# TARGET_PRECISION_FAKE: biến cấu hình/hằng số của notebook
TARGET_PRECISION_FAKE = 0.975

# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = Path(os.environ.get("FAKE_REVIEWS_PROJECT_ROOT", "/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews"))
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / "artifacts" / "features"
# MODEL_DIR: biến cấu hình/hằng số của notebook
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
# ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
# PREDICTION_DIR: biến cấu hình/hằng số của notebook
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
# REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
# REPORT_FIGURE_DIR: biến cấu hình/hằng số của notebook
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
# PROCESSED_DIR: biến cấu hình/hằng số của notebook
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# for: vòng lặp — for directory in [REPORT_FIGURE_DIR]:
for directory in [REPORT_FIGURE_DIR]:
    # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    directory.mkdir(parents=True, exist_ok=True)

# RAW_FEATURE_PATHS: biến cấu hình/hằng số của notebook
RAW_FEATURE_PATHS = {
    # "train": FEATURE_DIR / "features_raw_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "features_raw_train.npy",
    # "val": FEATURE_DIR / "features_raw_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "features_raw_val.npy",
    # "test": FEATURE_DIR / "features_raw_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "features_raw_test.npy",
# }: đóng khối từ điển
}
# LABEL_PATHS: biến cấu hình/hằng số của notebook
LABEL_PATHS = {
    # "train": FEATURE_DIR / "labels_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "labels_train.npy",
    # "val": FEATURE_DIR / "labels_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "labels_val.npy",
    # "test": FEATURE_DIR / "labels_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "labels_test.npy",
# }: đóng khối từ điển
}
# FEATURE_METADATA_PATH: biến cấu hình/hằng số của notebook
FEATURE_METADATA_PATH = FEATURE_DIR / "feature_metadata.json"


# configure_seed_artifacts: thiết lập đường dẫn artifact theo seed
def configure_seed_artifacts(seed: int) -> int:
    # """Canonical paths for seed 42; isolated subfolders for other seeds.""": thực thi lệnh Python
    """Canonical paths for seed 42; isolated subfolders for other seeds."""
    # global SEED, PREDICTION_DIR, ENSEMBLE_DIR, REPORT_TABLE_DIR, MODEL_DIR: thực thi lệnh Python
    global SEED, PREDICTION_DIR, ENSEMBLE_DIR, REPORT_TABLE_DIR, MODEL_DIR
    # SEED: biến cấu hình/hằng số của notebook
    SEED = int(seed)
    # if: điều kiện — if SEED == 42:
    if SEED == 42:
        # prediction_dir = ...: dự đoán nhãn/xác suất
        prediction_dir = PROJECT_ROOT / "artifacts" / "predictions"
        # ensemble_dir = ...: gán giá trị cho biến ensemble dir
        ensemble_dir = PROJECT_ROOT / "artifacts" / "ensemble"
        # report_table_dir = ...: gán giá trị cho biến report table dir
        report_table_dir = PROJECT_ROOT / "reports" / "tables"
        # model_dir = ...: gán giá trị cho biến model dir
        model_dir = PROJECT_ROOT / "artifacts" / "models"
    # else: nhánh còn lại của điều kiện
    else:
        # tag = ...: gán giá trị cho biến tag
        tag = f"seed_{SEED}"
        # prediction_dir = ...: dự đoán nhãn/xác suất
        prediction_dir = PROJECT_ROOT / "artifacts" / "predictions" / tag
        # ensemble_dir = ...: gán giá trị cho biến ensemble dir
        ensemble_dir = PROJECT_ROOT / "artifacts" / "ensemble" / tag
        # report_table_dir = ...: gán giá trị cho biến report table dir
        report_table_dir = PROJECT_ROOT / "reports" / "tables" / tag
        # model_dir = ...: gán giá trị cho biến model dir
        model_dir = PROJECT_ROOT / "artifacts" / "models" / tag
    # for: vòng lặp — for directory in [prediction_dir, ensemble_dir, report_table
    for directory in [prediction_dir, ensemble_dir, report_table_dir, model_dir]:
        # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
        directory.mkdir(parents=True, exist_ok=True)
    # PREDICTION_DIR: biến cấu hình/hằng số của notebook
    PREDICTION_DIR = prediction_dir
    # ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
    ENSEMBLE_DIR = ensemble_dir
    # REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
    REPORT_TABLE_DIR = report_table_dir
    # MODEL_DIR: biến cấu hình/hằng số của notebook
    MODEL_DIR = model_dir
    # return: trả kết quả từ hàm
    return SEED


# set_global_seed: cố định seed random và numpy
def set_global_seed(seed: int) -> None:
    # random.seed(seed): cố định seed random
    random.seed(seed)
    # np.random.seed(seed): cố định seed numpy
    np.random.seed(seed)


# set_torch_seed: hàm xử lý set torch seed
def set_torch_seed(seed: int) -> None:
    # import torch: deep learning PyTorch
    import torch

    # torch.manual_seed(seed): cố định seed torch
    torch.manual_seed(seed)
    # if: điều kiện — if torch.cuda.is_available():
    if torch.cuda.is_available():
        # torch.cuda.manual_seed_all(seed): thực thi lệnh Python
        torch.cuda.manual_seed_all(seed)


# configure_seed_artifacts(SEEDS[0]): thực thi lệnh Python
configure_seed_artifacts(SEEDS[0])
# set_global_seed(SEEDS[0]): tạo tập hợp
set_global_seed(SEEDS[0])


# utc_now: định nghĩa hàm utc now
def utc_now() -> str:
    # return: trả kết quả từ hàm
    return datetime.now(timezone.utc).isoformat()


# ensure_package: import hoặc pip install package
def ensure_package(import_name: str, pip_name: str | None = None):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)
    # except: xử lý ngoại lệ — except ImportError:
    except ImportError:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or impor...: thực thi lệnh Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)


# read_json: đọc file JSON
def read_json(path: Path, default=None):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # return: trả kết quả từ hàm
        return default if default is not None else {}
    # with: context manager — with path.open("r", encoding="utf-8") as file:
    with path.open("r", encoding="utf-8") as file:
        # return: parse nội dung JSON
        return json.load(file)


# load_raw_arrays: nạp feature/label .npy từ Phase 2
def load_raw_arrays():
    # missing = ...: kiểm tra file/thư mục tồn tại
    missing = [str(path) for path in list(RAW_FEATURE_PATHS.values()) + list(LABEL_PATHS.values()) if not path.exists()]
    # if: điều kiện — if missing:
    if missing:
        # raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(miss...: ghi dictionary ra JSON
        raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(missing, indent=2))
    # X: biến cấu hình/hằng số của notebook
    X = {split: np.load(path).astype(np.float32, copy=False) for split, path in RAW_FEATURE_PATHS.items()}
    # y = ...: nạp mảng từ file .npy
    y = {split: np.load(path).astype(np.int64, copy=False) for split, path in LABEL_PATHS.items()}
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # if: điều kiện — if X[split].ndim != 2:
        if X[split].ndim != 2:
            # raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}"): ném lỗi và dừng cell
            raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}")
        # if: điều kiện — if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
        if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
            # raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[spl...: ném lỗi và dừng cell
            raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[split].shape}")
        # if: điều kiện — if not np.isfinite(X[split]).all():
        if not np.isfinite(X[split]).all():
            # raise ValueError(f"Non-finite raw features in {split}"): ném lỗi và dừng cell
            raise ValueError(f"Non-finite raw features in {split}")
    # return: trả kết quả từ hàm
    return X, y


# safe_roc_auc: ROC-AUC an toàn
def safe_roc_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(roc_auc_score(y_true, prob_fake))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# safe_pr_auc: PR-AUC an toàn
def safe_pr_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(average_precision_score(y_true, prob_fake, pos_label=FAKE_LABEL))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# evaluate_predictions: tính metric classification từ xác suất
def evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=DEFAULT_THRESHOLD, threshold_strategy="default_0.5"):
    # y_true = ...: ép kiểu dữ liệu cột
    y_true = np.asarray(y_true).astype(int)
    # prob_fake = ...: ép kiểu dữ liệu cột
    prob_fake = np.asarray(prob_fake).astype(float)
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob_fake >= threshold).astype(int)
    # precision, recall, f1, support = precision_recall_fscore_support(: thực thi lệnh Python
    precision, recall, f1, support = precision_recall_fscore_support(
        # y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0: thực thi lệnh Python
        y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0
    # ): đóng ngoặc gọi hàm
    )
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL...: thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL]).ravel()
    # return: trả kết quả từ hàm
    return {
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "seed": int(SEED),: ép kiểu số nguyên
        "seed": int(SEED),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "model_variant": model_variant,: thực thi lệnh Python
        "model_variant": model_variant,
        # "threshold": float(threshold),: ép kiểu số thực
        "threshold": float(threshold),
        # "threshold_strategy": threshold_strategy,: ép kiểu chuỗi
        "threshold_strategy": threshold_strategy,
        # "accuracy": float(accuracy_score(y_true, y_pred)),: ép kiểu số thực
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),: ép kiểu số thực
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # "precision_fake": float(precision[1]),: ép kiểu số thực
        "precision_fake": float(precision[1]),
        # "recall_fake": float(recall[1]),: ép kiểu số thực
        "recall_fake": float(recall[1]),
        # "f1_fake": float(f1[1]),: ép kiểu số thực
        "f1_fake": float(f1[1]),
        # "support_real": int(support[0]),: ép kiểu số nguyên
        "support_real": int(support[0]),
        # "support_fake": int(support[1]),: ép kiểu số nguyên
        "support_fake": int(support[1]),
        # "roc_auc": safe_roc_auc(y_true, prob_fake),: thực thi lệnh Python
        "roc_auc": safe_roc_auc(y_true, prob_fake),
        # "pr_auc": safe_pr_auc(y_true, prob_fake),: thực thi lệnh Python
        "pr_auc": safe_pr_auc(y_true, prob_fake),
        # "brier_score": float(brier_score_loss(y_true, prob_fake)),: ép kiểu số thực
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        # "tn": int(tn),: ép kiểu số nguyên
        "tn": int(tn),
        # "fp": int(fp),: ép kiểu số nguyên
        "fp": int(fp),
        # "fn": int(fn),: ép kiểu số nguyên
        "fn": int(fn),
        # "tp": int(tp),: ép kiểu số nguyên
        "tp": int(tp),
    # }: đóng khối từ điển
    }


# save_probability: lưu xác suất fake ra file .npy
def save_probability(prob_fake, model_variant, split):
    # path = ...: gán giá trị cho biến path
    path = PREDICTION_DIR / f"{model_variant}_{split}_prob.npy"
    # np.save(path, np.asarray(prob_fake, dtype=np.float32)): lưu mảng numpy ra file .npy
    np.save(path, np.asarray(prob_fake, dtype=np.float32))
    # return: trả kết quả từ hàm
    return str(path)


# write_metrics: ghi bảng metric ra CSV
def write_metrics(prob_map, y, model_variant, output_name):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split=split, model_variant=model_variant)
        # row["probability_path"] = save_probability(prob_map[split], model_variant, split...: thực thi lệnh Python
        row["probability_path"] = save_probability(prob_map[split], model_variant, split)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # path = ...: gán giá trị cho biến path
    path = REPORT_TABLE_DIR / output_name
    # df.to_csv(path, index=False): ghi DataFrame ra file CSV
    df.to_csv(path, index=False)
    # display(df): hiển thị bảng/kết quả trên notebook
    display(df)
    # print("Saved metrics:", path): in thông tin ra console
    print("Saved metrics:", path)
    # return: trả kết quả từ hàm
    return df, path


In [ ]:
# build_base_candidates: hàm xử lý build base candidates
def build_base_candidates():
    # return: trả kết quả từ hàm
    return {
        # "phase5_lgbm_raw": {split: PREDICTION_DIR / f"phase5_lgbm_raw_{split}_prob.npy" ...: thực thi lệnh Python
        "phase5_lgbm_raw": {split: PREDICTION_DIR / f"phase5_lgbm_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
        # "phase5_xgb_raw": {split: PREDICTION_DIR / f"phase5_xgb_raw_{split}_prob.npy" fo...: thực thi lệnh Python
        "phase5_xgb_raw": {split: PREDICTION_DIR / f"phase5_xgb_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
        # "phase5_mlp_raw": {split: PREDICTION_DIR / f"phase5_mlp_raw_{split}_prob.npy" fo...: thực thi lệnh Python
        "phase5_mlp_raw": {split: PREDICTION_DIR / f"phase5_mlp_raw_{split}_prob.npy" for split in ["train", "val", "test"]},
        # "phase5_cnn_bilstm_sequence": {split: PREDICTION_DIR / f"phase5_cnn_bilstm_seque...: thực thi lệnh Python
        "phase5_cnn_bilstm_sequence": {split: PREDICTION_DIR / f"phase5_cnn_bilstm_sequence_{split}_prob.npy" for split in ["train", "val", "test"]},
    # }: đóng khối từ điển
    }


# load_available_probability_maps: hàm xử lý load available probability maps
def load_available_probability_maps(candidate_paths=None):
    # candidate_paths = ...: gán giá trị cho biến candidate paths
    candidate_paths = candidate_paths or build_base_candidates()
    # available = ...: gán giá trị cho biến available
    available = {}
    # skipped = ...: gán giá trị cho biến skipped
    skipped = []
    # for: vòng lặp — for candidate, split_paths in candidate_paths.items():
    for candidate, split_paths in candidate_paths.items():
        # if: điều kiện — if all(path.exists() for path in split_paths.values()):
        if all(path.exists() for path in split_paths.values()):
            # available[candidate] = {split: np.load(path).astype(np.float32) for split, path ...: nạp mảng từ file .npy
            available[candidate] = {split: np.load(path).astype(np.float32) for split, path in split_paths.items()}
        # else: nhánh còn lại của điều kiện
        else:
            # skipped.append(candidate): thực thi lệnh Python
            skipped.append(candidate)
    # if: điều kiện — if not available:
    if not available:
        # raise FileNotFoundError("No complete candidate probability sets found. Run base ...: ném lỗi và dừng cell
        raise FileNotFoundError("No complete candidate probability sets found. Run base model notebooks first.")
    # print("Available candidates:", sorted(available)): sắp xếp danh sách
    print("Available candidates:", sorted(available))
    # if: điều kiện — if skipped:
    if skipped:
        # print("Skipped incomplete candidates:", skipped): in thông tin ra console
        print("Skipped incomplete candidates:", skipped)
    # return: trả kết quả từ hàm
    return available


# threshold_sweep: hàm xử lý threshold sweep
def threshold_sweep(y_true, prob_fake, model_variant, split="val"):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
    for threshold in np.round(np.arange(0.01, 1.00, 0.01), 2):
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=float(threshold), threshold_strategy="threshold_sweep")
        # row["target_precision_fake"] = TARGET_PRECISION_FAKE: thực thi lệnh Python
        row["target_precision_fake"] = TARGET_PRECISION_FAKE
        # row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE): ép kiểu boolean
        row["target_met"] = bool(row["precision_fake"] >= TARGET_PRECISION_FAKE)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # return: trả kết quả từ hàm
    return pd.DataFrame(rows)


In [ ]:
# _, y = load_raw_arrays(): thực thi lệnh Python
_, y = load_raw_arrays()
# WEIGHT_STEP: biến cấu hình/hằng số của notebook
WEIGHT_STEP = 0.05
# for: vòng lặp — for SEED in SEEDS:
for SEED in SEEDS:
    # configure_seed_artifacts(SEED): thực thi lệnh Python
    configure_seed_artifacts(SEED)
    # set_global_seed(SEED): tạo tập hợp
    set_global_seed(SEED)
    # print(f"\n=== seed={SEED} ==="): in thông tin ra console
    print(f"\n=== seed={SEED} ===")
    # rows, best_records, blend_prob_maps = [], [], {}: đếm số phần tử
    rows, best_records, blend_prob_maps = [], [], {}
    # base_probs = ...: gán giá trị cho biến base probs
    base_probs = load_available_probability_maps()

    # simplex_weights: hàm xử lý simplex weights
    def simplex_weights(n, step=WEIGHT_STEP):
        # grid = ...: tạo dãy số cho vòng lặp
        grid = np.round(np.arange(0, 1 + step, step), 4)
        # for: vòng lặp — for weights in product(grid, repeat=n):
        for weights in product(grid, repeat=n):
            # if: điều kiện — if abs(sum(weights) - 1.0) <= 1e-6:
            if abs(sum(weights) - 1.0) <= 1e-6:
                # yield tuple(float(w) for w in weights): ép kiểu số thực
                yield tuple(float(w) for w in weights)

    # for: vòng lặp — for size in range(2, len(base_probs) + 1):
    for size in range(2, len(base_probs) + 1):
        # for: vòng lặp — for combo in combinations(sorted(base_probs), size):
        for combo in combinations(sorted(base_probs), size):
            # combo_key = ...: gán giá trị cho biến combo key
            combo_key = "+".join(combo)
            # best_row, best_prob_map = None, None: thực thi lệnh Python
            best_row, best_prob_map = None, None
            # for: vòng lặp — for weights in simplex_weights(len(combo)):
            for weights in simplex_weights(len(combo)):
                # prob_map = ...: gán giá trị cho biến prob map
                prob_map = {}
                # for: vòng lặp — for split in ["train", "val", "test"]:
                for split in ["train", "val", "test"]:
                    # blended = ...: ép kiểu số thực
                    blended = np.zeros_like(next(iter(base_probs.values()))[split], dtype=np.float32)
                    # for: vòng lặp — for model_name, weight in zip(combo, weights):
                    for model_name, weight in zip(combo, weights):
                        # blended += weight * base_probs[model_name][split]: đếm số phần tử
                        blended += weight * base_probs[model_name][split]
                    # prob_map[split] = blended.astype(np.float32): ép kiểu dữ liệu cột
                    prob_map[split] = blended.astype(np.float32)
                # name = ...: đếm số phần tử
                name = f"phase5_weighted_blend_{combo_key}_{'_'.join(f'{w:.2f}' for w in weights)}"
                # row = ...: dự đoán nhãn/xác suất
                row = evaluate_predictions(y["val"], prob_map["val"], "val", name)
                # row["base_models"] = combo_key: thực thi lệnh Python
                row["base_models"] = combo_key
                # row["weights_json"] = json.dumps(dict(zip(combo, weights)), sort_keys=True): ghi dictionary ra JSON
                row["weights_json"] = json.dumps(dict(zip(combo, weights)), sort_keys=True)
                # row["candidate_type"] = "weighted_blend": đếm số phần tử
                row["candidate_type"] = "weighted_blend"
                # rows.append(row): thực thi lệnh Python
                rows.append(row)
                # if: điều kiện — if best_row is None or (row["macro_f1"], row["roc_auc"], row
                if best_row is None or (row["macro_f1"], row["roc_auc"], row["precision_fake"], row["recall_fake"]) > (best_row["macro_f1"], best_row["roc_auc"], best_row["precision_fake"], best_row["recall_fake"]):
                    # best_row, best_prob_map = row, prob_map: thực thi lệnh Python
                    best_row, best_prob_map = row, prob_map
            # if: điều kiện — if best_row is not None:
            if best_row is not None:
                # best_records.append(best_row): thực thi lệnh Python
                best_records.append(best_row)
                # blend_prob_maps[combo_key] = best_prob_map: đếm số phần tử
                blend_prob_maps[combo_key] = best_prob_map

    # sweep_df = ...: gán giá trị cho biến sweep df
    sweep_df = pd.DataFrame(rows)
    # best_df = ...: gán giá trị cho biến best df
    best_df = pd.DataFrame(best_records).sort_values(["macro_f1", "roc_auc", "precision_fake", "recall_fake"], ascending=[False, False, False, False])
    # sweep_df.to_csv(REPORT_TABLE_DIR / "phase5_weighted_blending_sweep.csv", index=F...: ghi DataFrame ra file CSV
    sweep_df.to_csv(REPORT_TABLE_DIR / "phase5_weighted_blending_sweep.csv", index=False)
    # best_df.to_csv(REPORT_TABLE_DIR / "phase5_weighted_blending_best.csv", index=Fal...: ghi DataFrame ra file CSV
    best_df.to_csv(REPORT_TABLE_DIR / "phase5_weighted_blending_best.csv", index=False)
    # selected_key = ...: gán giá trị cho biến selected key
    selected_key = best_df.iloc[0]["base_models"]
    # selected_prob_map = ...: đếm số phần tử
    selected_prob_map = blend_prob_maps[selected_key]
    # metrics_df, metrics_path = write_metrics(selected_prob_map, y, "phase5_weighted_...: đếm số phần tử
    metrics_df, metrics_path = write_metrics(selected_prob_map, y, "phase5_weighted_blend", "phase5_weighted_blend_metrics.csv")
    # with: context manager — with (ENSEMBLE_DIR / "phase5_weighted_blend_metadata.json").
    with (ENSEMBLE_DIR / "phase5_weighted_blend_metadata.json").open("w", encoding="utf-8") as file:
        # json.dump({"seed": SEED, "selected_base_models": selected_key, "selected_weights...: ghi dictionary ra JSON
        json.dump({"seed": SEED, "selected_base_models": selected_key, "selected_weights_json": best_df.iloc[0]["weights_json"], "weight_step": WEIGHT_STEP, "metrics_path": str(metrics_path)}, file, indent=2)
    # display(best_df.head(20)): hiển thị bảng/kết quả trên notebook
    display(best_df.head(20))
    # print(f"Finished weighted blend for seed={SEED}"): in thông tin ra console
    print(f"Finished weighted blend for seed={SEED}")
    # gc.collect(): giải phóng bộ nhớ
    gc.collect()



=== seed=42 ===
Available candidates: ['phase5_cnn_bilstm_sequence', 'phase5_lgbm_raw', 'phase5_mlp_raw', 'phase5_xgb_raw']


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-10T09:37:36.379021+00:00,42,train,phase5_weighted_blend,0.5,default_0.5,0.976941,0.976053,0.984976,0.958272,...,17677,12246,0.998504,0.997880,0.020775,17498,179,511,11735,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-10T09:37:36.790241+00:00,42,val,phase5_weighted_blend,0.5,default_0.5,0.947450,0.944983,0.970383,0.899009,...,3789,2624,0.975749,0.975820,0.048631,3717,72,265,2359,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-10T09:37:37.134772+00:00,42,test,phase5_weighted_blend,0.5,default_0.5,0.945891,0.943320,0.969872,0.895579,...,3789,2624,0.976900,0.977018,0.047740,3716,73,274,2350,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/phase5_weighted_blend_metrics.csv


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,base_models,weights_json,candidate_type
2,2026-06-10T09:37:09.852231+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.947450,0.944983,0.970383,0.899009,...,0.975749,0.975820,0.048631,3717,72,265,2359,phase5_cnn_bilstm_sequence+phase5_xgb_raw,"{""phase5_cnn_bilstm_sequence"": 0.5, ""phase5_xg...",weighted_blend
7,2026-06-10T09:37:14.273243+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.947450,0.944983,0.970383,0.899009,...,0.975749,0.975820,0.048631,3717,72,265,2359,phase5_cnn_bilstm_sequence+phase5_lgbm_raw+pha...,"{""phase5_cnn_bilstm_sequence"": 0.5, ""phase5_lg...",weighted_blend
8,2026-06-10T09:37:16.189167+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.947450,0.944983,0.970383,0.899009,...,0.975749,0.975820,0.048631,3717,72,265,2359,phase5_cnn_bilstm_sequence+phase5_mlp_raw+phas...,"{""phase5_cnn_bilstm_sequence"": 0.5, ""phase5_ml...",weighted_blend
10,2026-06-10T09:37:32.750263+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.947450,0.944983,0.970383,0.899009,...,0.975749,0.975820,0.048631,3717,72,265,2359,phase5_cnn_bilstm_sequence+phase5_lgbm_raw+pha...,"{""phase5_cnn_bilstm_sequence"": 0.5, ""phase5_lg...",weighted_blend
6,2026-06-10T09:37:12.484876+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.945891,0.943444,0.963747,0.901677,...,0.975670,0.974405,0.048312,3700,89,258,2366,phase5_cnn_bilstm_sequence+phase5_lgbm_raw+pha...,"{""phase5_cnn_bilstm_sequence"": 0.55, ""phase5_l...",weighted_blend
0,2026-06-10T09:37:09.347020+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.945423,0.942825,0.969447,0.894817,...,0.976171,0.975688,0.048715,3715,74,276,2348,phase5_cnn_bilstm_sequence+phase5_lgbm_raw,"{""phase5_cnn_bilstm_sequence"": 0.55, ""phase5_l...",weighted_blend
1,2026-06-10T09:37:09.630424+00:00,42,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.939966,0.937426,0.948338,0.902439,...,0.973293,0.970437,0.049775,3660,129,256,2368,phase5_cnn_bilstm_sequence+phase5_mlp_raw,"{""phase5_cnn_bilstm_sequence"": 0.65, ""phase5_m...",weighted_blend
5,2026-06-10T09:37:10.618340+00:00,42,val,phase5_weighted_blend_phase5_mlp_raw+phase5_xg...,0.5,default_0.5,0.917667,0.912814,0.960457,0.833079,...,0.957709,0.955979,0.067752,3699,90,438,2186,phase5_mlp_raw+phase5_xgb_raw,"{""phase5_mlp_raw"": 0.5, ""phase5_xgb_raw"": 0.5}",weighted_blend
9,2026-06-10T09:37:16.825821+00:00,42,val,phase5_weighted_blend_phase5_lgbm_raw+phase5_m...,0.5,default_0.5,0.917667,0.912814,0.960457,0.833079,...,0.957709,0.955979,0.067752,3699,90,438,2186,phase5_lgbm_raw+phase5_mlp_raw+phase5_xgb_raw,"{""phase5_lgbm_raw"": 0.0, ""phase5_mlp_raw"": 0.5...",weighted_blend
3,2026-06-10T09:37:10.093016+00:00,42,val,phase5_weighted_blend_phase5_lgbm_raw+phase5_m...,0.5,default_0.5,0.916888,0.911941,0.960776,0.830793,...,0.956636,0.954415,0.069059,3700,89,444,2180,phase5_lgbm_raw+phase5_mlp_raw,"{""phase5_lgbm_raw"": 0.45, ""phase5_mlp_raw"": 0.55}",weighted_blend


Finished weighted blend for seed=42

=== seed=123 ===
Available candidates: ['phase5_cnn_bilstm_sequence', 'phase5_lgbm_raw', 'phase5_xgb_raw']
Skipped incomplete candidates: ['phase5_mlp_raw']


...

Finished weighted blend for seed=123

=== seed=456 ===
Available candidates: ['phase5_cnn_bilstm_sequence', 'phase5_lgbm_raw', 'phase5_xgb_raw']
Skipped incomplete candidates: ['phase5_mlp_raw']


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-10T09:37:52.825804+00:00,456,train,phase5_weighted_blend,0.5,default_0.5,0.987100,0.986646,0.987825,0.980565,...,17677,12246,0.999723,0.999601,0.007683,17529,148,238,12008,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-10T09:37:52.853102+00:00,456,val,phase5_weighted_blend,0.5,default_0.5,0.942772,0.940320,0.953395,0.904345,...,3789,2624,0.975997,0.975123,0.048094,3673,116,251,2373,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-10T09:37:52.870619+00:00,456,test,phase5_weighted_blend,0.5,default_0.5,0.942772,0.940273,0.955591,0.902058,...,3789,2624,0.976583,0.976217,0.046665,3679,110,257,2367,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/seed_456/phase5_weighted_blend_metrics.csv


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,base_models,weights_json,candidate_type
3,2026-06-10T09:37:52.218092+00:00,456,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.942772,0.940320,0.953395,0.904345,...,0.975997,0.975123,0.048094,3673,116,251,2373,phase5_cnn_bilstm_sequence+phase5_lgbm_raw+pha...,"{""phase5_cnn_bilstm_sequence"": 0.6, ""phase5_lg...",weighted_blend
1,2026-06-10T09:37:49.786584+00:00,456,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.942461,0.939963,0.954454,0.902439,...,0.976084,0.975558,0.048049,3676,113,256,2368,phase5_cnn_bilstm_sequence+phase5_xgb_raw,"{""phase5_cnn_bilstm_sequence"": 0.55, ""phase5_x...",weighted_blend
0,2026-06-10T09:37:49.508505+00:00,456,val,phase5_weighted_blend_phase5_cnn_bilstm_sequen...,0.5,default_0.5,0.942461,0.939963,0.954454,0.902439,...,0.975910,0.974955,0.048240,3676,113,256,2368,phase5_cnn_bilstm_sequence+phase5_lgbm_raw,"{""phase5_cnn_bilstm_sequence"": 0.6, ""phase5_lg...",weighted_blend
2,2026-06-10T09:37:50.015215+00:00,456,val,phase5_weighted_blend_phase5_lgbm_raw+phase5_x...,0.5,default_0.5,0.910338,0.904307,0.970170,0.805640,...,0.954634,0.954316,0.071664,3724,65,510,2114,phase5_lgbm_raw+phase5_xgb_raw,"{""phase5_lgbm_raw"": 0.4, ""phase5_xgb_raw"": 0.6}",weighted_blend


Finished weighted blend for seed=456
